In [1]:
# importation des librairies
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import zscore as stats_zscore

In [2]:
RAW    = "/mnt/user-data/outputs/data/raw/"
ROUTES = "/mnt/user-data/outputs/data/routes/"
CLEAN  = "/mnt/user-data/outputs/data/clean/"
os.makedirs(CLEAN, exist_ok=True)
 

In [3]:
LOG = []

In [4]:
def log(dataset, action, detail, avant=None, apres=None):
    LOG.append({"dataset":dataset,"action":action,"detail":detail,
                "lignes_avant":avant,"lignes_apres":apres,
                "timestamp":datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
    flag = "✅" if (avant is None or avant == apres) else f"🔧 ({avant}→{apres})"
    print(f"  {flag} [{dataset}] {action} — {detail}")

In [5]:
def rapport_qualite(df, nom):
    print(f"\n  📋 {nom} : {df.shape[0]}L × {df.shape[1]}C | "
          f"Nulls={df.isnull().sum().sum()} | Doublons={df.duplicated().sum()}")

In [30]:
# 1. LPI
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  1/5 — LPI Global Ranking")
print("="*55)
 
lpi = pd.read_csv("LPI_Global_Ranking.csv")
n0 = len(lpi)
rapport_qualite(lpi, "LPI brut")
 
lpi.columns = lpi.columns.str.strip().str.lower().str.replace(' ', '_')
log("LPI", "Colonnes", "snake_case normalisé")
 
score_cols = ['lpi_score','customs','infrastructure','logistics_quality',
              'tracking_tracing','timeliness','international_shipments']
for col in score_cols:
    before = lpi[(lpi[col] < 1) | (lpi[col] > 5)].shape[0]
    lpi[col] = lpi[col].clip(1, 5)
    log("LPI", "Validation", f"{col} clampé [1-5] — {before} valeurs corrigées")
 
lpi['lpi_category'] = pd.cut(lpi['lpi_score'],
    bins=[0, 2.5, 3.0, 3.5, 4.0, 5.1],
    labels=['Très faible','Faible','Moyen','Bon','Excellent'])
 
lpi['resilience_score'] = (
    lpi['infrastructure'] * 0.30 +
    lpi['logistics_quality'] * 0.25 +
    lpi['timeliness'] * 0.25 +
    lpi['customs'] * 0.20
).round(3)
 
lpi['above_world_avg'] = (lpi['lpi_score'] >= lpi['lpi_score'].mean()).astype(int)
log("LPI", "Enrichissement", "lpi_category + resilience_score + above_world_avg ajoutés")
 
dups = lpi.duplicated(subset=['country','iso3']).sum()
if dups > 0:
    lpi = lpi.drop_duplicates(subset=['country','iso3'])
    log("LPI", "Doublons", f"{dups} supprimés", n0, len(lpi))
else:
    log("LPI", "Doublons", "Aucun")
 




  1/5 — LPI Global Ranking

  📋 LPI brut : 50L × 11C | Nulls=0 | Doublons=0
  ✅ [LPI] Colonnes — snake_case normalisé
  ✅ [LPI] Validation — lpi_score clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — customs clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — infrastructure clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — logistics_quality clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — tracking_tracing clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — timeliness clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Validation — international_shipments clampé [1-5] — 0 valeurs corrigées
  ✅ [LPI] Enrichissement — lpi_category + resilience_score + above_world_avg ajoutés
  ✅ [LPI] Doublons — Aucun


In [31]:
lpi.to_csv("lpi_clean.csv", index=False)

In [32]:
log("LPI", "Export", f"lpi_clean.csv — {len(lpi)} lignes")
rapport_qualite(lpi, "LPI clean")

  ✅ [LPI] Export — lpi_clean.csv — 50 lignes

  📋 LPI clean : 50L × 14C | Nulls=0 | Doublons=0


In [9]:
# 2. WPI
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  2/5 — World Port Index")
print("="*55)
 
wpi = pd.read_csv("world_port_index.csv")
n0 = len(wpi)
rapport_qualite(wpi, "WPI brut")
 
wpi.columns = wpi.columns.str.strip().str.lower().str.replace(' ', '_')
 
# Coordonnées
bad_geo = wpi[(wpi['latitude'].abs() > 90) | (wpi['longitude'].abs() > 180)].shape[0]
log("WPI", "Coordonnées", f"{bad_geo} coordonnées invalides détectées")
 
# max_depth
wpi['max_depth'] = pd.to_numeric(wpi['max_depth'], errors='coerce')
depth_nulls = wpi['max_depth'].isnull().sum()
if depth_nulls > 0:
    med = wpi['max_depth'].median()
    wpi['max_depth'] = wpi['max_depth'].fillna(med)
    log("WPI", "Imputation", f"max_depth : {depth_nulls} nulls → médiane {med:.1f}m")
else:
    log("WPI", "Validation", "max_depth : aucun null")



  2/5 — World Port Index

  📋 WPI brut : 40L × 17C | Nulls=0 | Doublons=0
  ✅ [WPI] Coordonnées — 0 coordonnées invalides détectées
  ✅ [WPI] Validation — max_depth : aucun null


In [10]:
# Encodage Yes/No → 1/0 (déjà string)
for col in ['anchorage', 'drydock', 'railway']:
    wpi[col] = wpi[col].astype(str).str.strip().map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    log("WPI", "Encodage", f"{col} → 1/0")

  ✅ [WPI] Encodage — anchorage → 1/0
  ✅ [WPI] Encodage — drydock → 1/0
  ✅ [WPI] Encodage — railway → 1/0


In [33]:
# Scores composites
wpi['risk_score'] = (
    (1 - wpi['max_depth'] / wpi['max_depth'].max()) * 0.40 +
    wpi['congestion_index'] * 0.60
).round(3)
 
wpi['risk_category'] = pd.cut(wpi['risk_score'],
    bins=[0, 0.3, 0.5, 0.7, 1.01],
    labels=['Faible','Modéré','Élevé','Critique'])
 
wpi['infrastructure_score'] = (
    (wpi['max_depth'] / wpi['max_depth'].max()) * 0.5 +
    wpi['anchorage'] * 0.2 +
    wpi['drydock'] * 0.15 +
    wpi['railway'] * 0.15
).round(3)
 
log("WPI", "Enrichissement", "risk_score + risk_category + infrastructure_score ajoutés")
 
dups = wpi.duplicated(subset=['port_name','country']).sum()
if dups > 0:
    wpi = wpi.drop_duplicates(subset=['port_name','country'])
    log("WPI", "Doublons", f"{dups} supprimés", n0, len(wpi))
else:
    log("WPI", "Doublons", "Aucun")
 
wpi.to_csv("ports_clean.csv", index=False)
log("WPI", "Export", f"ports_clean.csv — {len(wpi)} lignes")
rapport_qualite(wpi, "WPI clean")

  ✅ [WPI] Enrichissement — risk_score + risk_category + infrastructure_score ajoutés
  ✅ [WPI] Doublons — Aucun
  ✅ [WPI] Export — ports_clean.csv — 40 lignes

  📋 WPI clean : 40L × 20C | Nulls=0 | Doublons=0


In [13]:
# 3. TRADE FLOWS
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  3/5 — Trade Flows")
print("="*55)
 
trade = pd.read_csv("trade_flows_global.csv")
n0 = len(trade)
rapport_qualite(trade, "Trade brut")
 
trade.columns = trade.columns.str.strip().str.lower().str.replace(' ', '_')
 
# Valeurs négatives
neg = (trade['trade_value_usd'] <= 0).sum()
if neg > 0:
    trade = trade[trade['trade_value_usd'] > 0]
    log("Trade", "Filtrage", f"{neg} valeurs <= 0 supprimées", n0, len(trade))
else:
    log("Trade", "Validation", "trade_value_usd : toutes > 0")


  3/5 — Trade Flows

  📋 Trade brut : 3040L × 8C | Nulls=0 | Doublons=0
  ✅ [Trade] Validation — trade_value_usd : toutes > 0


In [14]:
# Années
hors = trade[~trade['year'].isin([2021,2022,2023,2024])].shape[0]
if hors > 0:
    trade = trade[trade['year'].isin([2021,2022,2023,2024])]
    log("Trade", "Filtrage", f"{hors} lignes hors période", n0, len(trade))
else:
    log("Trade", "Validation", f"Années : {sorted(trade['year'].unique())}")
 
# Auto-commerce
auto = (trade['reporter_iso'] == trade['partner_iso']).sum()
if auto > 0:
    trade = trade[trade['reporter_iso'] != trade['partner_iso']]
    log("Trade", "Filtrage", f"{auto} auto-commerces supprimés", n0, len(trade))

  ✅ [Trade] Validation — Années : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [35]:
# Enrichissement
trade['trade_value_billion'] = (trade['trade_value_usd'] / 1e9).round(3)
trade['trade_intensity'] = pd.cut(trade['trade_value_usd'],
    bins=[0, 1e9, 10e9, 50e9, float('inf')],
    labels=['Faible','Modéré','Fort','Très fort'])
 
exports = trade[trade['flow']=='Export'][['year','reporter','partner','trade_value_usd']].rename(columns={'trade_value_usd':'export_val'})
imports = trade[trade['flow']=='Import'][['year','reporter','partner','trade_value_usd']].rename(columns={'trade_value_usd':'import_val'})
balance = pd.merge(exports, imports, on=['year','reporter','partner'], how='outer').fillna(0)
balance['trade_balance']     = (balance['export_val'] - balance['import_val']).round(0)
balance['dependency_ratio']  = (balance['import_val'] / (balance['export_val'] + balance['import_val'] + 1)).round(3)
 
trade = pd.merge(trade, balance[['year','reporter','partner','trade_balance','dependency_ratio']],
                 on=['year','reporter','partner'], how='left')
 
log("Trade", "Enrichissement", "trade_value_billion + trade_intensity + trade_balance + dependency_ratio")
 
dups = trade.duplicated(subset=['year','reporter','partner','flow']).sum()
if dups > 0:
    trade = trade.drop_duplicates(subset=['year','reporter','partner','flow'])
    log("Trade", "Doublons", f"{dups} supprimés", n0, len(trade))
else:
    log("Trade", "Doublons", "Aucun")
 
trade.to_csv("trade_clean.csv", index=False)
log("Trade", "Export", f"trade_clean.csv — {len(trade)} lignes")
rapport_qualite(trade, "Trade clean")

  ✅ [Trade] Enrichissement — trade_value_billion + trade_intensity + trade_balance + dependency_ratio
  ✅ [Trade] Doublons — Aucun
  ✅ [Trade] Export — trade_clean.csv — 3040 lignes

  📋 Trade clean : 3040L × 14C | Nulls=0 | Doublons=0


In [17]:
# 4. SUEZ
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  4/5 — Suez Canal Traffic")
print("="*55)
 
suez = pd.read_csv("suez_canal_traffic.csv")
n0 = len(suez)
rapport_qualite(suez, "Suez brut")
 
suez.columns = suez.columns.str.strip().str.lower().str.replace(' ', '_')
 
# Dates
suez['date'] = pd.to_datetime(suez['date'], errors='coerce')
bad_dates = suez['date'].isnull().sum()
if bad_dates > 0:
    suez = suez.dropna(subset=['date'])
    log("Suez", "Dates", f"{bad_dates} dates invalides supprimées", n0, len(suez))
else:
    log("Suez", "Dates", "Toutes les dates sont valides")


  4/5 — Suez Canal Traffic

  📋 Suez brut : 72L × 9C | Nulls=0 | Doublons=0
  ✅ [Suez] Dates — Toutes les dates sont valides


In [18]:
# Cohérence north + south
suez['computed_total'] = suez['northbound'] + suez['southbound']
incoherents = (abs(suez['computed_total'] - suez['total_transits']) > 5).sum()
if incoherents > 0:
    suez['total_transits'] = suez['computed_total']
    log("Suez", "Correction", f"{incoherents} totaux recalculés")
else:
    log("Suez", "Validation", "Cohérence northbound + southbound = total ✅")
suez = suez.drop(columns=['computed_total'])
 
# Outliers via z-score (conservés car événements réels)
zs = np.abs(stats_zscore(suez['total_transits']))
n_outliers = (zs > 3).sum()
log("Suez", "Outliers", f"{n_outliers} valeurs z>3 détectées — conservées (événements réels)")

  ✅ [Suez] Validation — Cohérence northbound + southbound = total ✅
  ✅ [Suez] Outliers — 1 valeurs z>3 détectées — conservées (événements réels)


In [36]:
# Enrichissement temporel
suez['quarter']          = suez['date'].dt.quarter
suez['quarter_label']    = 'Q' + suez['quarter'].astype(str)
suez['year_month']       = suez['date'].dt.to_period('M').astype(str)
suez['mom_change_pct']   = suez['total_transits'].pct_change().round(4)
suez['rolling_3m']       = suez['total_transits'].rolling(3, min_periods=1).mean().round(1)
suez['rolling_12m']      = suez['total_transits'].rolling(12, min_periods=1).mean().round(1)
suez['is_crisis']        = (suez['event_flag'] != 'Normal').astype(int)
normal_avg               = suez[suez['event_flag']=='Normal']['total_transits'].mean()
suez['transit_vs_normal']= (suez['total_transits'] / normal_avg).round(3)
 
log("Suez", "Enrichissement", "quarter + rolling_3m/12m + is_crisis + transit_vs_normal ajoutés")
 
suez.to_csv("suez_clean.csv", index=False)
log("Suez", "Export", f"suez_clean.csv — {len(suez)} lignes")
rapport_qualite(suez, "Suez clean")

  ✅ [Suez] Enrichissement — quarter + rolling_3m/12m + is_crisis + transit_vs_normal ajoutés
  ✅ [Suez] Export — suez_clean.csv — 72 lignes

  📋 Suez clean : 72L × 17C | Nulls=1 | Doublons=0


In [21]:
# 5. ROUTES
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  5/5 — Routes OSRM")
print("="*55)
 
routes = pd.read_csv("routes_ports_villes.csv")
n0 = len(routes)
rapport_qualite(routes, "Routes brut")
 
routes.columns = routes.columns.str.strip().str.lower().str.replace(' ', '_')
 
# Distances négatives
neg = (routes['distance_km'] <= 0).sum()
if neg > 0:
    routes = routes[routes['distance_km'] > 0]
    log("Routes", "Filtrage", f"{neg} distances <= 0 supprimées", n0, len(routes))
else:
    log("Routes", "Validation", "Toutes distances > 0")


  5/5 — Routes OSRM

  📋 Routes brut : 30L × 12C | Nulls=0 | Doublons=0
  ✅ [Routes] Validation — Toutes distances > 0


In [22]:
# Vitesse cohérente
routes['speed_kmh'] = (routes['distance_km'] / (routes['duration_minutes'] / 60)).round(1)
bad_speed = ((routes['speed_kmh'] < 10) | (routes['speed_kmh'] > 150)).sum()
if bad_speed > 0:
    mask = (routes['speed_kmh'] < 10) | (routes['speed_kmh'] > 150)
    routes.loc[mask, 'duration_minutes'] = (routes.loc[mask, 'distance_km'] / 70 * 60).round(0).astype(int)
    routes['speed_kmh'] = (routes['distance_km'] / (routes['duration_minutes'] / 60)).round(1)
    log("Routes", "Correction", f"{bad_speed} durées recalculées à 70km/h")
else:
    log("Routes", "Validation", f"Vitesses OK : {routes['speed_kmh'].min():.0f}–{routes['speed_kmh'].max():.0f} km/h")


  ✅ [Routes] Validation — Vitesses OK : 21–105 km/h


In [37]:
# Enrichissement
routes['duration_hours']     = (routes['duration_minutes'] / 60).round(2)
routes['cost_estimate_usd']  = (routes['distance_km'] * 2.1).round(0)
routes['co2_kg']             = (routes['distance_km'] * 0.062).round(1)
routes['corridor']           = routes['distance_km'].apply(
    lambda d: 'Court (<100km)' if d < 100 else ('Moyen (100-500km)' if d < 500 else 'Long (>500km)'))
 
log("Routes", "Enrichissement", "duration_hours + cost_estimate_usd + co2_kg + corridor ajoutés")
 
dups = routes.duplicated(subset=['start_port','destination_city']).sum()
if dups > 0:
    routes = routes.drop_duplicates(subset=['start_port','destination_city'])
    log("Routes", "Doublons", f"{dups} supprimés", n0, len(routes))
else:
    log("Routes", "Doublons", "Aucun")
 
routes.to_csv("routes_clean.csv", index=False)
log("Routes", "Export", f"routes_clean.csv — {len(routes)} lignes")
rapport_qualite(routes, "Routes clean")

  ✅ [Routes] Enrichissement — duration_hours + cost_estimate_usd + co2_kg + corridor ajoutés
  ✅ [Routes] Doublons — Aucun
  ✅ [Routes] Export — routes_clean.csv — 30 lignes

  📋 Routes clean : 30L × 17C | Nulls=0 | Doublons=0


In [38]:
# RAPPORT FINAL
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  RAPPORT FINAL ETL")
print("="*55)
 
fichiers_clean = {
    "lpi_clean.csv":    pd.read_csv("lpi_clean.csv"),
    "ports_clean.csv":  pd.read_csv("ports_clean.csv"),
    "trade_clean.csv":  pd.read_csv("trade_clean.csv"),
    "suez_clean.csv":   pd.read_csv("suez_clean.csv"),
    "routes_clean.csv": pd.read_csv("routes_clean.csv"),
}
 
print(f"\n  {'Fichier':<22} {'Lignes':>7} {'Colonnes':>9} {'Nulls':>7} {'Doublons':>9}")
print("  " + "-"*56)
for fname, df in fichiers_clean.items():
    print(f"  {fname:<22} {len(df):>7} {len(df.columns):>9} {df.isnull().sum().sum():>7} {df.duplicated().sum():>9}")
 
pd.DataFrame(LOG).to_csv("etl_log.csv", index=False)
print(f"\n  ✅ Log ETL → etl_log.csv ({len(LOG)} actions enregistrées)")
print("  ✅ ETL terminé avec succès !")


  RAPPORT FINAL ETL

  Fichier                 Lignes  Colonnes   Nulls  Doublons
  --------------------------------------------------------
  lpi_clean.csv               50        14       0         0
  ports_clean.csv             40        20       0         0
  trade_clean.csv           3040        14       0         0
  suez_clean.csv              72        17       1         0
  routes_clean.csv            30        17       0         0

  ✅ Log ETL → etl_log.csv (78 actions enregistrées)
  ✅ ETL terminé avec succès !
